# 27d · Vá `dataset_v3_*` đã có — thêm nhãn 14 + `cohort_study_level.parquet`

**Dùng khi bạn đã chạy `27c` bản cũ rồi và không muốn chạy lại.** Notebook này **không**
đụng tới `.npy` (7.2 GB giữ nguyên, `mtime` không đổi) và **không** đọc cache CLS.

Nó thêm vào mỗi thư mục arm đang có:

| file | nội dung |
|---|---|
| `index.parquet` | ghi lại, **thêm cột `Support_Devices`** (nhãn thứ 14) |
| `class_freq.pkl` | `labels`/`class_freq`/`num_train` cho **14** nhãn |
| `cohort_study_level.parquet` | bảng cohort đầy đủ + 14 nhãn **⊕ `x_cxr` của arm đó** |
| `columns.json` | ghi lại, `labels` → 14 |

## Vì sao không cần pool lại

`x_cxr` của mỗi arm **đã nằm sẵn trong 768 cột cuối** của `v2_embedding.npy`. Notebook chỉ
đọc lát đó ra (`mmap`), không cần `cxr_cls.npy`, không cần `view_map`, không tính lại
trọng số. `C4` đối chiếu lát đó với `v1_onehot.npy` để chắc chắn đọc đúng chỗ.

## ⚠ `Support Devices` là confound đã biết

`01` **cố ý** bỏ nhãn này (`DROP_LABELS`, lý do: *"not a pathology (tubes/lines/pacemakers)"*).
Nó là ống, dây, máy tạo nhịp — tương quan mạnh với "bệnh nhân nặng / nằm ICU", nên model
rất dễ dùng làm đường tắt và điểm ở các nhãn khác trông đẹp lên mà không phải vì đọc được
phổi. Giữ nó thì **báo cáo riêng**; đừng gộp vào macro rồi so với số cũ 13 nhãn — hai con
số không cùng thang đo.

Lấy nhãn theo **đúng quy ước `01`**: nguồn BigQuery `physionet-data.mimic_cxr_jpg.chexpert`,
tên cột đọc từ schema (không đoán), U-Ones `-1 → 1`, `NaN → 0`, `int8`, LEFT join
`study_id`. **Split giữ nguyên** — `01` sinh split trên 13 nhãn; tái sinh sẽ phá tính so
sánh với `dataset_v3` và mọi kết quả cũ.

## Chạy trên máy gì

CPU thuần, không cần GPU. RAM đỉnh ~3 GB. ~6–10 phút, gần như toàn bộ là ghi
`cohort_study_level.parquet` (~750 MB/arm, tổng ~3 GB).

## C0 · Config

In [1]:
import os, json, pickle
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = os.environ.get('PLANB_OUT', './planB_out')

OUT_DIR = os.path.join(DATA_DIR, 'cohort_v2')
ARMS    = ['dataset_v3_73', 'dataset_v3_64', 'dataset_v3_55', 'dataset_v3_frontal']

WRITE_COHORT_PARQUET     = True
COHORT_PARQUET_WITH_XCXR = True    # False -> cohort parquet khong kem x_cxr (nhe hon nhieu)
ADD_SUPPORT_DEVICES      = True
EXPECT_N_LABELS          = 14

PROJECT  = 'mimic-ehr-v310'                 # nhu 01 C0
CXR      = 'physionet-data.mimic_cxr_jpg'   # nhu 01 C0 -> bang {CXR}.chexpert
U_POLICY = 'ones'                           # nhu 01 C0
SUPPORT_DEVICES_SRC = os.environ.get('SD_SRC')   # None -> lay tu BigQuery

COHORT_SRC = os.path.join(OUT_DIR, 'cohort_study_level.parquet')

for a in ARMS:
    d = os.path.join(DATA_DIR, a)
    assert os.path.isdir(d), f'{d} khong co -> chay 27c truoc'
    for f in ['index.parquet', 'columns.json', 'v2_embedding.npy']:
        assert os.path.exists(os.path.join(d, f)), f'{a}/{f} thieu'
if WRITE_COHORT_PARQUET:
    assert os.path.exists(COHORT_SRC), f'{COHORT_SRC} khong co -> chay 02'

print('se va:', ARMS)
print('nguon cohort:', COHORT_SRC)
print('Support_Devices tu:', SUPPORT_DEVICES_SRC or f'BigQuery {CXR}.chexpert')

Mounted at /content/drive
se va: ['dataset_v3_73', 'dataset_v3_64', 'dataset_v3_55', 'dataset_v3_frontal']
nguon cohort: /content/drive/MyDrive/MIMIC_CXR/planB_out/cohort_v2/cohort_study_level.parquet
Support_Devices tu: BigQuery physionet-data.mimic_cxr_jpg.chexpert


## C1 · Auth BigQuery — **chạy cell này**, mỗi session một lần

Colab **không** có service account mặc định, nên nếu bỏ qua bước này thì `bigquery.Client()`
rơi xuống credential của GCE metadata và chết bằng
`RefreshError: Failed to retrieve ... metadata.google.internal ... Status: 404`.
Đó không phải lỗi quyền — chỉ là chưa có credential nào.

Cell dưới in ra một URL. Mở URL → đăng nhập → copy mã xác thực → dán vào ô input hiện ra.
**Không** dùng `google.colab.auth.authenticate_user()`.

*(Bỏ qua cả C1 nếu bạn đặt `SUPPORT_DEVICES_SRC` ở `C0` — lúc đó không đụng BigQuery.)*

In [2]:
!gcloud auth application-default login --no-launch-browser
!gcloud auth application-default set-quota-project {PROJECT}

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=d6Zdg3rRhVNwj75VtKtrX9OZVYM3QJ&prompt=consent&token_usage=remote&access_type=offline&code_challenge=RhVq-w20TIGUAUqUTtqdQxTKty7jG_F2EgTFzhCfL_Q&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0ATsMZqArlfsLoP98ldWw-9xpTRuM9EFOFnZICTy1g_LnAiTXGUDMzfTMjUMigYZEPTmkpg

Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Ca

### C1-bis · Xác nhận credential — biến `RefreshError` thành câu tiếng người

Ép thư viện dùng **file ADC vừa tạo**, chứ không để nó tự dò rồi rơi xuống GCE metadata.

In [7]:
if ADD_SUPPORT_DEVICES and SUPPORT_DEVICES_SRC is None:
    # gcloud tren Colab dat CLOUDSDK_CONFIG=/content/.config -> ADC KHONG nam o ~/.config/gcloud
    _cands = [os.environ.get('GOOGLE_APPLICATION_CREDENTIALS'),
              (os.path.join(os.environ['CLOUDSDK_CONFIG'], 'application_default_credentials.json')
               if os.environ.get('CLOUDSDK_CONFIG') else None),
              '/content/.config/application_default_credentials.json',
              os.path.expanduser('~/.config/gcloud/application_default_credentials.json')]
    _adc = next((p for p in _cands if p and os.path.exists(p)), None)
    if _adc:
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = _adc     # chan roi xuong GCE metadata
        print('ADC:', _adc)
    else:
        print('! khong thay file ADC o cac cho quen thuoc:')
        for p in _cands:
            if p:
                print('   -', p)
        print('  -> van thu google.auth.default(); cong ben duoi moi la cho quyet dinh')

    import google.auth
    _creds, _proj = google.auth.default()
    _kind = type(_creds).__module__
    print(f'credential: {type(_creds).__name__} ({_kind}) | quota project: {_proj}')
    assert 'compute_engine' not in _kind, (
        'dang dung credential cua GCE metadata -> cell gcloud chua chay thanh cong. '
        'Chay lai va dan ma xac thuc.')
    print('COANG · credential la ADC nguoi dung, khong phai GCE metadata  OK')
else:
    print('bo qua auth (dung SUPPORT_DEVICES_SRC hoac ADD_SUPPORT_DEVICES=False)')


ADC: /content/.config/application_default_credentials.json
credential: Credentials (google.oauth2.credentials) | quota project: None
COANG · credential la ADC nguoi dung, khong phai GCE metadata  OK


## C2 · Đọc trạng thái hiện có + lấy nhãn 14

Trật tự chủ lấy từ `index.parquet` của arm đầu tiên, và **bắt buộc cả 4 arm phải giống
nhau** — nếu không thì không thể dùng chung một vector nhãn.

In [8]:
SPEC0 = json.load(open(os.path.join(DATA_DIR, ARMS[0], 'columns.json')))
DX    = int(SPEC0['x_cxr']['dim'])
_lab  = list(SPEC0['labels'])
assert len(_lab) != 40, 'columns.json co 40 nhan -> nham nhanh CXR-LT (28+), khong phai cohort_v2'
# IDEMPOTENT: da va roi (hoac va do vi Colab rot) thi chay lai van phai duoc.
if _lab and _lab[-1] == 'Support_Devices':
    print('! cac arm da duoc va truoc do -> chay lai, se ghi de (idempotent)')
    _lab = _lab[:-1]
LABELS13 = _lab
assert len(LABELS13) == 13, (f'{len(LABELS13)} nhan sau khi bo Support_Devices, mong doi 13 '
                             '-> columns.json khong phai cua cohort_v2')

IDX0 = pd.read_parquet(os.path.join(DATA_DIR, ARMS[0], 'index.parquet'))
N    = len(IDX0)
SID  = IDX0.study_id.to_numpy()
IS_TRAIN = (IDX0.split.to_numpy() == 'train')
assert (IDX0.row.to_numpy() == np.arange(N)).all()

# COÅNG: 4 arm phai cung trat tu study
for a in ARMS[1:]:
    _i = pd.read_parquet(os.path.join(DATA_DIR, a, 'index.parquet'))
    assert len(_i) == N and (_i.study_id.to_numpy() == SID).all(), f'{a}: trat tu study lech {ARMS[0]}'
print(f'COÅNG · {len(ARMS)} arm cung trat tu study ({N:,} hang)  OK')

LABELS = LABELS13 + (['Support_Devices'] if ADD_SUPPORT_DEVICES else [])
C = len(LABELS)
assert C == EXPECT_N_LABELS, f'{C} nhan, mong doi {EXPECT_N_LABELS}'

SD = None
if ADD_SUPPORT_DEVICES:
    if 'Support_Devices' in IDX0.columns:
        print('! index.parquet DA co Support_Devices -> se ghi de bang ban lay moi')
    if SUPPORT_DEVICES_SRC:
        _t = (pd.read_parquet(SUPPORT_DEVICES_SRC) if SUPPORT_DEVICES_SRC.endswith('.parquet')
              else pd.read_csv(SUPPORT_DEVICES_SRC))
        _t = _t.rename(columns={c: 'sd' for c in _t.columns if c != 'study_id'})
        print(f'nguon: {SUPPORT_DEVICES_SRC}  ({len(_t):,} hang)')
    else:
        from google.cloud import bigquery
        client = bigquery.Client(project=PROJECT, location='US')
        _cols = [f.name for f in client.get_table(f'{CXR}.chexpert').schema]
        def _norm(c):
            return c.replace('_', ' ').strip().lower()
        _cand = [c for c in _cols if _norm(c) == 'support devices']
        assert len(_cand) == 1, f'khong tim thay dung 1 cot "support devices" trong {_cols}'
        print(f'cot BigQuery: `{_cand[0]}`')
        _t = client.query(f'SELECT study_id, `{_cand[0]}` AS sd FROM `{CXR}.chexpert`').to_dataframe()
        print(f'chexpert: {len(_t):,} hang')

    print('gia tri THO truoc U-policy:', dict(_t.sd.value_counts(dropna=False)))
    _v = _t.sd
    if U_POLICY == 'ones':
        _v = _v.replace(-1, 1)
    elif U_POLICY == 'zeros':
        _v = _v.replace(-1, 0)
    _t = _t.assign(sd=_v.fillna(0).astype('int8')).drop_duplicates('study_id')

    _m = pd.Series(_t.sd.to_numpy(), index=_t.study_id.to_numpy()).reindex(SID)
    _miss = int(_m.isna().sum())
    print(f'COÅNG · phu het cohort: thieu {_miss} / {N:,}  {"OK" if _miss == 0 else "FAIL"}')
    assert _miss == 0, f'{_miss} study khong co trong chexpert -> nguon sai hoac khac cohort'
    SD = _m.to_numpy().astype('int8')
    assert set(np.unique(SD)) <= {0, 1}, f'gia tri {set(np.unique(SD))}, phai 0/1'
    print(f'\nSupport_Devices: prevalence cohort {SD.mean():.4f} | train {SD[IS_TRAIN].mean():.4f}'
          f'  ({int(SD.sum()):,} duong tinh)')
    print('  -> nhan RAT PHO BIEN so voi 13 nhan kia. Bao cao RIENG, dung gop vao macro')
    print('     roi so voi so cu 13 nhan.')

Y = np.column_stack([IDX0[l].to_numpy('int8') for l in LABELS13]
                    + ([SD] if ADD_SUPPORT_DEVICES else [])).astype('int8')
CLASS_FREQ = Y[IS_TRAIN].sum(0).astype('int64')
NUM_TRAIN  = int(IS_TRAIN.sum())
assert (CLASS_FREQ > 0).all(), 'co nhan khong xuat hien o train -> DB Loss se chia 0'
print(f'\nclass_freq moi ({C} nhan, train n={NUM_TRAIN:,}):')
print(pd.DataFrame({'label': LABELS, 'train_pos': CLASS_FREQ,
                    'train_prev': (CLASS_FREQ / NUM_TRAIN).round(4)})
        .sort_values('train_prev').to_string(index=False))

COHORT = None
if WRITE_COHORT_PARQUET:
    COHORT = pd.read_parquet(COHORT_SRC).sort_values('study_id').reset_index(drop=True)
    assert len(COHORT) == N and (COHORT.study_id.to_numpy() == SID).all(), \
        'cohort_study_level.parquet lech voi index.parquet -> khac cohort'
    _mi = [l for l in LABELS13 if l not in COHORT.columns]
    assert not _mi, f'cohort nguon thieu nhan: {_mi}'
    assert not any(c.startswith('x_cxr_') for c in COHORT.columns), 'cohort nguon da co x_cxr_*'
    print(f'\ncohort nguon: {COHORT.shape[0]:,} hang x {COHORT.shape[1]} cot')

COÅNG · 4 arm cung trat tu study (211,130 hang)  OK
cot BigQuery: `Support_Devices`
chexpert: 227,827 hang
gia tri THO truoc U-policy: {nan: np.int64(157546), 1.0: np.int64(66558), 0.0: np.int64(3486), -1.0: np.int64(237)}
COÅNG · phu het cohort: thieu 0 / 211,130  OK

Support_Devices: prevalence cohort 0.2990 | train 0.2992  (63,135 duong tinh)
  -> nhan RAT PHO BIEN so voi 13 nhan kia. Bao cao RIENG, dung gop vao macro
     roi so voi so cu 13 nhan.

class_freq moi (14 nhan, train n=147,791):
                     label  train_pos  train_prev
             Pleural_Other       1763      0.0119
                  Fracture       3212      0.0217
               Lung_Lesion       4761      0.0322
              Pneumothorax       7555      0.0511
             Consolidation       9934      0.0672
Enlarged_Cardiomediastinum      10848      0.0734
                 Pneumonia      22603      0.1529
                     Edema      26553      0.1797
              Cardiomegaly      33305      0.2254


## C3 · Vá từng arm — `.npy` **không bị đụng tới**

`x_cxr` đọc từ 768 cột cuối của `v2_embedding.npy` bằng `mmap`, nên không nạp cả 700 MB
vào RAM và không ghi lại gì. Cột `frontal_is_fallback` (nếu arm đó có) được giữ nguyên.

In [9]:
XCOLS = [f'x_cxr_{i}' for i in range(DX)]

def patch_arm(arm):
    d = os.path.join(DATA_DIR, arm)
    sp = json.load(open(os.path.join(d, 'columns.json')))
    v2 = sp['variants']['v2_embedding']
    p_npy = os.path.join(d, v2['file'])

    mt_before = os.path.getmtime(p_npy)
    X  = np.load(p_npy, mmap_mode='r')
    assert X.shape == (N, len(v2['columns'])), (arm, X.shape)
    assert v2['columns'][-DX:] == XCOLS, f'{arm}: {DX} cot cuoi khong phai x_cxr_*'
    xc = np.ascontiguousarray(X[:, -DX:]).astype('float32')
    del X
    assert not np.isnan(xc).any(), f'{arm}: NaN trong x_cxr'

    # ---- index.parquet: giu moi cot cu, them/ghi de Support_Devices ----
    ix = pd.read_parquet(os.path.join(d, 'index.parquet'))
    if ADD_SUPPORT_DEVICES:
        ix['Support_Devices'] = SD
    assert [c for c in ix.columns if c in LABELS] == LABELS, f'{arm}: thu tu nhan lech'
    ix.to_parquet(os.path.join(d, 'index.parquet'), index=False)

    # ---- class_freq.pkl ----
    with open(os.path.join(d, 'class_freq.pkl'), 'wb') as fh:
        pickle.dump(dict(labels=LABELS, class_freq=CLASS_FREQ.tolist(), num_train=NUM_TRAIN), fh)

    # ---- cohort_study_level.parquet ----
    if WRITE_COHORT_PARQUET:
        coh = COHORT.copy()
        if ADD_SUPPORT_DEVICES:
            coh['Support_Devices'] = SD
        if 'frontal_is_fallback' in ix.columns:
            coh['frontal_is_fallback'] = ix.frontal_is_fallback.to_numpy()
        if COHORT_PARQUET_WITH_XCXR:
            coh = pd.concat([coh.reset_index(drop=True),
                             pd.DataFrame(xc, columns=XCOLS)], axis=1)
        p = os.path.join(d, 'cohort_study_level.parquet')
        coh.to_parquet(p, compression='snappy', index=False)
        print(f'    cohort_study_level.parquet {coh.shape}  {os.path.getsize(p)/1e6:.0f} MB')
        del coh

    # ---- columns.json ----
    sp['labels']   = LABELS
    sp['n_labels'] = C
    sp['label_note'] = (
        'Support_Devices lay lai tu physionet-data.mimic_cxr_jpg.chexpert voi U-Ones '
        '(-1->1, NaN->0), dung quy uoc cua 01. 01 co chu y bo no vi "not a pathology". '
        'Day la confound da biet (ICU/benh nang) -- bao cao RIENG, dung gop vao macro roi '
        'so voi so cu 13 nhan.') if ADD_SUPPORT_DEVICES else '13 nhan'
    sp['patched_by'] = '27d_patch_14labels_and_cohort.ipynb'
    sp['files'] = dict(
        index=f'index.parquet ({C} nhan)',
        class_freq=f'class_freq.pkl ({C} nhan)',
        cohort=('cohort_study_level.parquet (cohort day du'
                + (' + x_cxr cua arm nay)' if COHORT_PARQUET_WITH_XCXR else ')')
                if WRITE_COHORT_PARQUET else 'khong ghi'))
    json.dump(sp, open(os.path.join(d, 'columns.json'), 'w'), indent=1)

    assert os.path.getmtime(p_npy) == mt_before, f'{arm}: .npy bi ghi lai!'
    del xc
    return ix

for arm in ARMS:
    print(f'\n--- {arm} ---')
    _ix = patch_arm(arm)
    print(f'    index.parquet {_ix.shape} | class_freq.pkl {C} nhan | .npy khong doi')
print('\nva xong.')


--- dataset_v3_73 ---
    cohort_study_level.parquet (211130, 872)  1030 MB
    index.parquet (211130, 18) | class_freq.pkl 14 nhan | .npy khong doi

--- dataset_v3_64 ---
    cohort_study_level.parquet (211130, 872)  1030 MB
    index.parquet (211130, 18) | class_freq.pkl 14 nhan | .npy khong doi

--- dataset_v3_55 ---
    cohort_study_level.parquet (211130, 872)  1030 MB
    index.parquet (211130, 18) | class_freq.pkl 14 nhan | .npy khong doi

--- dataset_v3_frontal ---
    cohort_study_level.parquet (211130, 873)  1030 MB
    index.parquet (211130, 19) | class_freq.pkl 14 nhan | .npy khong doi

va xong.


## C4 · Nghiệm thu — nạp lại từ đĩa

Bốn thứ: nhãn đúng 14 ở mọi arm, `Support_Devices` giống nhau giữa các arm, `x_cxr` trong
`cohort_study_level.parquet` **khớp** lát cuối của `.npy`, và `.npy` không bị sứt.

In [10]:
print('=== nghiem thu ===')
ref_sd, ref_shape = None, {}
for arm in ARMS:
    d  = os.path.join(DATA_DIR, arm)
    sp = json.load(open(os.path.join(d, 'columns.json')))
    ix = pd.read_parquet(os.path.join(d, 'index.parquet'))
    cf = pickle.load(open(os.path.join(d, 'class_freq.pkl'), 'rb'))

    assert sp['labels'] == LABELS and len(sp['labels']) == C, f'{arm}: nhan trong columns.json sai'
    assert len(sp['labels']) != 40, f'{arm}: 40 nhan -> nham nhanh CXR-LT'
    for l in LABELS:
        assert l in ix.columns, f'{arm}: index.parquet thieu {l}'
    assert cf['labels'] == LABELS and len(cf['class_freq']) == C, f'{arm}: class_freq.pkl sai'
    assert cf['num_train'] == NUM_TRAIN
    if ADD_SUPPORT_DEVICES:
        sd = ix.Support_Devices.to_numpy()
        if ref_sd is None:
            ref_sd = sd
        else:
            assert (sd == ref_sd).all(), f'{arm}: Support_Devices khac arm khac'

    # COÅNG: .npy la ma tran FEATURE, khong chua nhan -> them nhan 14 khong the anh huong no
    for _v, _sv in sp['variants'].items():
        _feat = _sv['columns']
        _leak = [c for c in _feat if c in LABELS]
        assert not _leak, f'{arm}/{_v}: cot NHAN lot vao ma tran feature: {_leak}'
        _nx = sum(c.startswith('x_cxr_') for c in _feat)
        _nm = sum(c.startswith(tuple(f'{k}__' for k in sp.get('micro', {}).get('blocks', {})))
                  for c in _feat)
        print(f'    {_v:14s} {len(_feat):5d} cot = micro {_nm} + ehr {len(_feat)-_nm-_nx} '
              f'+ x_cxr {_nx}  | cot nhan: 0')

    line = f'  {arm:20s} nhan {len(sp["labels"])} | index {ix.shape}'
    if WRITE_COHORT_PARQUET:
        ch = pd.read_parquet(os.path.join(d, 'cohort_study_level.parquet'))
        assert len(ch) == N and (ch.study_id.to_numpy() == SID).all(), f'{arm}: cohort parquet lech'
        assert all(l in ch.columns for l in LABELS), f'{arm}: cohort parquet thieu nhan'
        if COHORT_PARQUET_WITH_XCXR:
            nx = sum(c.startswith('x_cxr_') for c in ch.columns)
            assert nx == DX, f'{arm}: cohort parquet co {nx} cot x_cxr, mong doi {DX}'
            V = np.load(os.path.join(d, sp['variants']['v2_embedding']['file']), mmap_mode='r')
            e = float(np.abs(ch[XCOLS].to_numpy('float32') - np.ascontiguousarray(V[:, -DX:])).max())
            assert e == 0.0, f'{arm}: x_cxr trong cohort parquet khac .npy ({e:.2e})'
            del V
        line += f' | cohort {ch.shape}'
        ref_shape[arm] = ch.shape
        del ch
    print(line + '   OK')

if ADD_SUPPORT_DEVICES:
    print(f'\nSupport_Devices giong nhau o ca {len(ARMS)} arm | prevalence {ref_sd.mean():.4f}')
print('\n=== file trong tung arm ===')
for arm in ARMS:
    d = os.path.join(DATA_DIR, arm)
    sz = sum(os.path.getsize(os.path.join(d, f)) for f in os.listdir(d))
    print(f'\n{arm}/   ({sz/1e9:.2f} GB)')
    for f in sorted(os.listdir(d)):
        print(f'    {f:34s} {os.path.getsize(os.path.join(d, f))/1e6:9.1f} MB')
print('\nTAT CA CONG DA QUA. .npy khong bi ghi lai.')

=== nghiem thu ===
    v1_onehot       1272 cot = micro 475 + ehr 29 + x_cxr 768  | cot nhan: 0
    v2_embedding     829 cot = micro 32 + ehr 29 + x_cxr 768  | cot nhan: 0
  dataset_v3_73        nhan 14 | index (211130, 18) | cohort (211130, 872)   OK
    v1_onehot       1272 cot = micro 475 + ehr 29 + x_cxr 768  | cot nhan: 0
    v2_embedding     829 cot = micro 32 + ehr 29 + x_cxr 768  | cot nhan: 0
  dataset_v3_64        nhan 14 | index (211130, 18) | cohort (211130, 872)   OK
    v1_onehot       1272 cot = micro 475 + ehr 29 + x_cxr 768  | cot nhan: 0
    v2_embedding     829 cot = micro 32 + ehr 29 + x_cxr 768  | cot nhan: 0
  dataset_v3_55        nhan 14 | index (211130, 18) | cohort (211130, 872)   OK
    v1_onehot       1272 cot = micro 475 + ehr 29 + x_cxr 768  | cot nhan: 0
    v2_embedding     829 cot = micro 32 + ehr 29 + x_cxr 768  | cot nhan: 0
  dataset_v3_frontal   nhan 14 | index (211130, 19) | cohort (211130, 873)   OK

Support_Devices giong nhau o ca 4 arm | prevalen